# 20c 거리 지표로 연구하기 — 사진 → 지표 → 사람의 평가 → 회귀

<a href="https://colab.research.google.com/github/kloud80/urban_datamining_26/blob/main/W06_%EC%9D%B4%EB%AF%B8%EC%A7%80%EB%B6%84%EC%84%9D1-2_%ED%83%90%EC%A7%80%EC%99%80%EB%B6%84%ED%95%A0/notebooks/20c%20%EA%B1%B0%EB%A6%AC%EC%A7%80%ED%91%9C%EB%A1%9C_%EC%97%B0%EA%B5%AC%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`20b` 에서는 사진 **한 장**을 분할해 녹시율을 냈다. 연구는 거기서 끝나지 않는다. 사진 **여러 장**에서 지표를 뽑고,
그 지표가 **사람의 반응**(보행 시간, 만족도, 범죄…)을 설명하는지 통계로 확인한다.

이 노트북은 한양대 이수기 교수 연구실의 아래 연구를 **교실 크기로 줄여** 따라 한다.

> Lieu, Ha, Kim, Ki & Lee (2021). 『국토계획』 56(2) — 서울 Street View 400장을 240명이 2,400번 평가했다.
> DeepLabV3+ (Cityscapes 사전학습, mIoU 82.1%)로 요소 비율을, YOLOv3 로 보행자·차량 수를 뽑아 평가 점수와 회귀했다.
> 결과 — **식생·하늘·보도는 +, 벽·차량 수는 −**, 보행자 수는 '활기'와 가장 강한 + 관계.

| 단계 | 하는 일 | 분할의 종류 |
|---|---|---|
| 1 | 서울 거리 사진 18장 | — |
| 2 | 녹시율·하늘·건물·보도 **비율** | 시맨틱 분할 (DeepLabV3+ · Cityscapes) |
| 3 | 사람·차 **개수** | 판옵틱 분할 (Mask2Former · COCO) |
| 4 | **우리가 직접** "어느 길이 더 걷고 싶은가" 쌍대비교 | — |
| 5 | 평가 점수 ~ 지표 회귀 → 논문과 부호 비교 | — |

> 런타임: **GPU 권장** (런타임 → 런타임 유형 변경 → T4). CPU 면 2·3단계에 5~10분 걸린다.

## 1. 준비

In [ ]:
import os, io, glob, random, urllib.request, tarfile, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import torch

import matplotlib.font_manager as fm
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('TensorFlow', tf.__version__, '/ torch', torch.__version__, '/', device)

## 2. 서울 거리 사진 18장

Wikimedia Commons 의 공개 라이선스 사진이다(저작자·라이선스는 아래 표). 사람 눈높이에서 길을 따라 찍은 낮 사진만 골랐다.
연구에서는 이 자리에 **Street View** 를 쓴다 — 지점을 일정 간격(20~50m)으로 찍고 사방 4장씩 받는다. 구글 API 는 유료 키가 필요해서 여기서는 대신 공개 사진을 쓴다.

**자기 사진을 더하려면** `photos/` 폴더에 jpg 를 올리면 된다. 많을수록 6번 회귀가 믿을 만해진다.

In [ ]:
PHOTOS = [   # (주소, 라이선스, 저작자) — Wikimedia Commons
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/7/70/628_Samgakji_Exit_7.JPG/960px-628_Samgakji_Exit_7.JPG', 'CC BY-SA 3.0', '주행거리계'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/4/4e/Around_Samneung_Park.jpg/960px-Around_Samneung_Park.jpg', 'CC BY-SA 2.5', 'Jérôme Banal (Eden2004)'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/2/25/Back_street_%28%E9%90%98%E8%B7%AF%E3%80%81Seoul%29_-_panoramio.jpg/960px-Back_street_%28%E9%90%98%E8%B7%AF%E3%80%81Seoul%29_-_panoramio.jpg', 'CC BY 3.0', 'yamarhythm'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c2/Back_street_%28%E9%90%98%E8%B7%AF%EF%BC%93%E8%A1%97%EF%BC%89_-_panoramio.jpg/960px-Back_street_%28%E9%90%98%E8%B7%AF%EF%BC%93%E8%A1%97%EF%BC%89_-_panoramio.jpg', 'CC BY 3.0', 'yamarhythm'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/0/0c/Back_to_Seoul_%285177155091%29.jpg/960px-Back_to_Seoul_%285177155091%29.jpg', 'CC BY-SA 2.0', 'Lisa Risager'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/1/15/Bonghwasan_Seouruiryowon_Station_20140228_171922.JPG/960px-Bonghwasan_Seouruiryowon_Station_20140228_171922.JPG', 'CC BY-SA 3.0', '안우석'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b7/Bonghwasan_Seouruiryowon_Station_20140228_172227.JPG/960px-Bonghwasan_Seouruiryowon_Station_20140228_172227.JPG', 'CC BY-SA 3.0', '안우석'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/5/5e/Bonghwasan_Seouruiryowon_Station_20140228_172954.JPG/960px-Bonghwasan_Seouruiryowon_Station_20140228_172954.JPG', 'CC BY-SA 3.0', '안우석'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/1/1e/Bukchon-ro_12-gil.jpg/960px-Bukchon-ro_12-gil.jpg', 'CC BY-SA 3.0', 'Adbar'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/3/3e/Downtown_Seoul_%281510625286%29.jpg/960px-Downtown_Seoul_%281510625286%29.jpg', 'CC BY 2.0', 'Francisco Anzola'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/8/86/DSC_2639_%2835437116202%29.jpg/960px-DSC_2639_%2835437116202%29.jpg', 'CC BY 2.0', 'chelsom tsai from TAIPEI, Taiwam'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/2/2b/Gangam_006.JPG/960px-Gangam_006.JPG', 'CC BY-SA 3.0', 'myself (User:Piotrus)'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/6/63/Gusan_Station_Exit_3.JPG/960px-Gusan_Station_Exit_3.JPG', 'CC BY 3.0', '쉰지식인'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/2/24/Jongno%2C_Seoul%2C_South_Korea_01.jpg/960px-Jongno%2C_Seoul%2C_South_Korea_01.jpg', 'KOGL Type 1', 'KOREA TOURISM ORGANIZATION (한국관광공사)'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b7/Bike_ride_in_my_neighborhood_in_Seoul%2C_Korea_%2812920850545%29.jpg/960px-Bike_ride_in_my_neighborhood_in_Seoul%2C_Korea_%2812920850545%29.jpg', 'CC BY-SA 2.0', 'Michael Sean Gallagher from London, United Kingdom'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e5/Bike_ride_in_my_neighborhood_in_Seoul%2C_Korea_%2812920948993%29.jpg/960px-Bike_ride_in_my_neighborhood_in_Seoul%2C_Korea_%2812920948993%29.jpg', 'CC BY-SA 2.0', 'Michael Sean Gallagher from London, United Kingdom'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/3/3a/Dongil_road.jpg/960px-Dongil_road.jpg', 'CC BY-SA 3.0', 'Bigsmile20'),
    ('https://thumb.wikimedia.org/wikipedia/commons/thumb/0/01/Harley-Davidson_KNPA_in_Seoul%2C_2018.jpg/960px-Harley-Davidson_KNPA_in_Seoul%2C_2018.jpg', 'CC BY-SA 4.0', 'Kevin.B')
]

os.makedirs('photos', exist_ok=True)
for i, (url, lic, who) in enumerate(PHOTOS):
    path = f'photos/s{i:02d}.jpg'
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={'User-Agent': 'urban-datamining-course/1.0'})
        with urllib.request.urlopen(req, timeout=60) as r, open(path, 'wb') as f:
            f.write(r.read())

files = sorted(glob.glob('photos/*.jp*g'))
print(len(files), '장')
pd.DataFrame(PHOTOS, columns=['url', 'license', 'author']).assign(file=[f's{i:02d}' for i in range(len(PHOTOS))])[['file', 'license', 'author']]

In [ ]:
fig, ax = plt.subplots(3, 6, figsize=(24, 9))
for a, f in zip(ax.ravel(), files):
    a.imshow(Image.open(f)); a.set_title(os.path.basename(f)[:-4], fontsize=13); a.axis('off')
for a in ax.ravel()[len(files):]:
    a.axis('off')
plt.tight_layout(); plt.show()

## 3. 시맨틱 분할 — **비율**을 잰다 (DeepLabV3+ · Cityscapes)

`20b` 와 같은 모델이다. 이수기 교수 연구실 논문들이 쓴 바로 그 사전학습 모델 —
*"we employed the Deeplabv3+ model (82.1% mIoU) … pre-trained with the Cityscapes dataset"* (Hipp, Lee, Ki & Kim, 2022).

픽셀마다 19개 클래스 중 하나를 칠하므로, 사진에서 **각 요소가 차지하는 비율**이 나온다. 같은 클래스의 나무 두 그루는 구분하지 않는다.

In [ ]:
TARBALL = 'deeplab_xception65_cityscapes.tar.gz'
if not os.path.exists(TARBALL):
    urllib.request.urlretrieve('http://download.tensorflow.org/models/deeplabv3_cityscapes_train_2018_02_06.tar.gz', TARBALL)
with tarfile.open(TARBALL) as tar:
    m = next(m for m in tar.getmembers() if 'frozen_inference_graph' in m.name)
    graph_def = tf.compat.v1.GraphDef.FromString(tar.extractfile(m).read())
_w = tf.compat.v1.wrap_function(lambda: tf.compat.v1.import_graph_def(graph_def, name=''), [])
deeplab = _w.prune('ImageTensor:0', 'SemanticPredictions:0')

CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light', 'traffic sign',
           'vegetation', 'terrain', 'sky', 'person', 'rider', 'car', 'truck', 'bus', 'train',
           'motorcycle', 'bicycle']

def semantic(img):
    img = img.convert('RGB')
    s = 513 / max(img.size)
    small = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)
    return deeplab(tf.constant(np.asarray(small)[None]))[0].numpy()

rows = []
for f in files:
    seg = semantic(Image.open(f))
    sh = np.bincount(seg.ravel(), minlength=19) / seg.size * 100
    c = CLASSES.index
    rows.append({'file': os.path.basename(f)[:-4],
                 '녹시율': sh[c('vegetation')] + sh[c('terrain')],
                 '하늘': sh[c('sky')],
                 '건물': sh[c('building')],
                 '벽·울타리': sh[c('wall')] + sh[c('fence')],
                 '보도': sh[c('sidewalk')],
                 '차도': sh[c('road')]})
sem = pd.DataFrame(rows).set_index('file').round(1)
sem

## 4. 판옵틱 분할 — **개수**를 센다 (Mask2Former · COCO)

시맨틱 분할은 사람이 3명이든 30명이든 'person 2%' 라고만 말한다. 연구에서 **보행자 수·차량 수**가 필요하면 물체를 하나하나 갈라야 한다.

* **시맨틱** — 픽셀마다 클래스. 비율을 잰다 (녹시율, 하늘 개방감)
* **인스턴스** — 물체마다 따로 마스크. 개수를 센다 (보행자 수, 차량 수)
* **판옵틱** — 둘을 합쳤다. 하늘·도로 같은 *배경*은 면적으로, 사람·차 같은 *사물*은 하나씩

Lieu et al. (2021)은 개수를 세려고 탐지 모델 YOLOv3 를 따로 붙였다. 여기서는 판옵틱 모델 하나로 둘을 같이 얻는다.

In [ ]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

M2F = 'facebook/mask2former-swin-tiny-coco-panoptic'
m2f_proc = AutoImageProcessor.from_pretrained(M2F)
m2f = Mask2FormerForUniversalSegmentation.from_pretrained(M2F).to(device).eval()   # 로그의 MISSING(layernorm)·UNEXPECTED 는 이 모델이 쓰지 않는 층이라 무시해도 된다
id2label = m2f.config.id2label

def panoptic(img, thr=0.5):
    img = img.convert('RGB')
    img.thumbnail((800, 800))
    inputs = m2f_proc(images=img, return_tensors='pt').to(device)
    with torch.no_grad():
        out = m2f(**inputs)
    res = m2f_proc.post_process_panoptic_segmentation(out, threshold=thr, target_sizes=[img.size[::-1]],
                                                   label_ids_to_fuse=set())[0]
    return img, res['segmentation'].cpu().numpy(), res['segments_info']

COUNT = {'person': '사람 수', 'car': '승용차 수', 'bus': '버스 수', 'truck': '트럭 수',
         'bicycle': '자전거 수', 'motorcycle': '오토바이 수'}
rows = []
for f in files:
    _, _, info = panoptic(Image.open(f))
    names = [id2label[s['label_id']] for s in info]
    rows.append({'file': os.path.basename(f)[:-4], **{v: names.count(k) for k, v in COUNT.items()}})
cnt = pd.DataFrame(rows).set_index('file')
cnt['차량 수'] = cnt[['승용차 수', '버스 수', '트럭 수']].sum(axis=1)
cnt

### 한 장을 세 가지로 본다

In [ ]:
SHOW = files[min(10, len(files) - 1)]
img, pan, info = panoptic(Image.open(SHOW))
seg = semantic(Image.open(SHOW))

rng = np.random.default_rng(0)
col = rng.integers(40, 255, size=(pan.max() + 2, 3)).astype(np.uint8)
pan_rgb = col[pan + 1]

fig, ax = plt.subplots(1, 3, figsize=(24, 6))
ax[0].imshow(img); ax[0].set_title('원본', fontsize=16)
ax[1].imshow(seg, cmap='tab20', vmin=0, vmax=19); ax[1].set_title('시맨틱 — 차가 몇 대든 한 색', fontsize=16)
ax[2].imshow(img); ax[2].imshow(pan_rgb, alpha=0.6)
for s in info:
    if id2label[s['label_id']] in COUNT:
        ys, xs = np.where(pan == s['id'])
        ax[2].text(xs.mean(), ys.mean(), id2label[s['label_id']], color='white', fontsize=11,
                   ha='center', bbox=dict(facecolor='black', alpha=0.6, pad=1))
ax[2].set_title('판옵틱 — 사람·차는 하나씩 따로', fontsize=16)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 5. 사람의 평가 — "어느 길이 더 걷고 싶은가?"

연구에서는 이 자리에 설문(보행 시간, 만족도)이나 행정 자료(범죄 건수)가 들어간다.
여기서는 **우리가 직접** 평가한다. MIT 의 Place Pulse 와 Lieu et al. (2021)이 쓴 **쌍대비교** 방식이다 — 두 장을 나란히 보고 하나를 고른다.
점수를 매기라고 하면 사람마다 기준이 다르지만, 둘 중 하나를 고르는 것은 일관되게 잘한다.

* 아래 셀을 실행하고 `RATER` 에 이름을 적은 뒤 **20번** 고른다.
* 끝나면 `responses_이름.csv` 가 저장된다. **반 전체의 CSV 를 모아 `photos/` 옆에 올리면** 6번이 전부 합쳐서 계산한다.

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output

RATER = '이름'          # ← 바꿀 것
N_PAIRS = 20

names = [os.path.basename(f)[:-4] for f in files]
pairs = [tuple(random.sample(names, 2)) for _ in range(N_PAIRS)]
answers = []
out = W.Output()

def show(k):
    with out:
        clear_output(wait=True)
        if k >= len(pairs):
            df = pd.DataFrame(answers, columns=['rater', 'left', 'right', 'choice'])
            df.to_csv(f'responses_{RATER}.csv', index=False)
            print(f'끝 — {len(df)}개 응답을 responses_{RATER}.csv 에 저장했다.')
            return
        a, b = pairs[k]
        fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
        for x, n in zip(ax, (a, b)):
            x.imshow(Image.open(f'photos/{n}.jpg')); x.axis('off')
        ax[0].set_title('왼쪽', fontsize=16); ax[1].set_title('오른쪽', fontsize=16)
        fig.suptitle(f'{k + 1}/{len(pairs)} — 어느 길을 더 걷고 싶은가?', fontsize=18)
        plt.show()

def pick(choice):
    def f(_):
        k = len(answers)
        if k < len(pairs):
            answers.append((RATER, *pairs[k], choice))
            show(k + 1)
    return f

btn = [W.Button(description=t, layout=W.Layout(width='160px', height='48px')) for t in ('◀ 왼쪽', '비슷하다', '오른쪽 ▶')]
for b, c in zip(btn, ('left', 'tie', 'right')):
    b.on_click(pick(c))
display(W.HBox(btn), out)
show(0)

## 6. 점수 만들기 — 쌍대비교 → 사진별 점수 (Bradley–Terry)

이긴 횟수만 세면, 강한 상대를 만난 사진이 손해를 본다. **Bradley–Terry** 모형은 "누구를 이겼는가"까지 따져
사진마다 하나의 점수를 매긴다(체스 레이팅과 같은 발상). 비슷하다는 양쪽에 반 승씩 준다.

In [ ]:
resp = pd.concat([pd.read_csv(p) for p in glob.glob('responses_*.csv')], ignore_index=True) \
       if glob.glob('responses_*.csv') else pd.DataFrame(columns=['rater', 'left', 'right', 'choice'])
print('응답', len(resp), '개 · 평가자', resp['rater'].nunique(), '명')

def bradley_terry(resp, items, iters=200):
    idx = {n: i for i, n in enumerate(items)}
    wins = np.zeros((len(items), len(items)))
    for _, r in resp.iterrows():
        i, j = idx[r.left], idx[r.right]
        if r.choice == 'left':   wins[i, j] += 1
        elif r.choice == 'right': wins[j, i] += 1
        else:                     wins[i, j] += .5; wins[j, i] += .5
    p = np.ones(len(items))
    for _ in range(iters):                              # MM 알고리즘 (Hunter, 2004)
        n = wins + wins.T
        den = (n / (p[:, None] + p[None, :] + 1e-12)).sum(1)
        p = (wins.sum(1) + 0.1) / (den + 1e-12)
        p /= np.exp(np.log(p).mean())
    return pd.Series(np.log(p), index=items, name='걷고 싶음')

if len(resp):
    score = bradley_terry(resp, names)
    score.sort_values().plot.barh(figsize=(8, 7), title='사진별 걷고 싶음 점수 (로그 척도)'); plt.show()
else:
    print('응답이 없다 — 5번에서 먼저 평가할 것.')

## 7. 회귀 — 무엇이 걷고 싶은 길을 만드는가

지표를 **표준화**(평균 0, 표준편차 1)해 계수를 서로 비교할 수 있게 한다.
그리고 논문의 부호와 맞는지 본다 — **식생·하늘·보도 +, 벽·차량 −**.

> ⚠️ 사진 18장에 변수 여러 개면 통계적으로 믿을 수 없다. 논문은 400장·2,400번 평가였다.
> 여기서는 **방향(부호)이 맞는지**만 보고, 반 전체 응답을 모을수록 좋아지는 것을 확인하자.

In [ ]:
import statsmodels.api as sm

X_COLS = ['녹시율', '하늘', '건물', '보도', '사람 수', '차량 수']
data = sem.join(cnt)
if len(resp):
    data = data.join(score)
    X = (data[X_COLS] - data[X_COLS].mean()) / data[X_COLS].std(ddof=0).replace(0, 1)
    ols = sm.OLS(data['걷고 싶음'], sm.add_constant(X)).fit()
    print(ols.summary().tables[1])

    PAPER = {'녹시율': '+', '하늘': '+', '보도': '+', '차량 수': '−'}
    coef = ols.params.drop('const')
    colors = ['#00B37A' if v > 0 else '#D93636' for v in coef]
    ax = coef.plot.barh(color=colors, figsize=(9, 4.5), title='표준화 계수 — 초록 +, 빨강 −')
    for y, k in enumerate(coef.index):
        if k in PAPER:
            ax.text(ax.get_xlim()[1], y, f'  논문 {PAPER[k]}', va='center', fontsize=12)
    ax.axvline(0, color='k', lw=1); plt.show()
else:
    print('응답이 없다 — 5번에서 먼저 평가할 것.')

## 정리와 해 볼 것

연구의 뼈대는 언제나 같다 — **사진 → 분할 → 지표 → 사람·도시의 반응 → 통계**. 분할 모델은 답이 아니라 **변수를 만드는 도구**다.

1. 반 전체의 `responses_*.csv` 를 모아 다시 돌려 보자. 계수의 부호가 논문과 가까워지는가?
2. 질문을 바꿔 보자 — "어느 길이 더 **안전해** 보이는가?" 범죄 연구(Hipp, Lee, Ki & Kim 2022)에서는 **벽**이 범죄와 −, 차량·사람이 + 였다.
3. 녹시율을 `vegetation` 만으로 바꾸면(잔디·화단 `terrain` 제외) 결과가 달라지는가?
4. 이 지표들이 **놓치는 것**은 무엇인가 — 그늘, 소음, 경사, 보도 폭, 상점의 종류…

### 참고
* Lieu, S., Ha, J., Kim, J. H., Ki, D., & Lee, S. (2021). 『국토계획』 56(2), 79–93. doi:10.17208/jkpa.2021.04.56.2.79
* Ki, D., Kim, J. H., & Lee, S. (2021). 『국토계획』 56(4), 194–211. doi:10.17208/jkpa.2021.08.56.4.194
* Hipp, J. R., Lee, S., Ki, D., & Kim, J. H. (2022). *Journal of Quantitative Criminology*, 38(3), 537–565.
* Dubey, A. et al. (2016). Deep Learning the City: Quantifying Urban Perception at a Global Scale. *ECCV*.
* Cheng, B. et al. (2022). Masked-attention Mask Transformer for Universal Image Segmentation (Mask2Former). *CVPR*.
* 사진: Wikimedia Commons — 2번 셀의 표에 사진마다 저작자와 라이선스(CC BY · CC BY-SA · KOGL 1유형)를 적었다.